<a href="https://colab.research.google.com/github/Anvay-A/Travel-Itinerary-Generator/blob/main/Travel_Itinerary_Generator_using_GenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  **GenAI Intensive Course Capstone Project**
#  **✈️ Travel Itinerary Generator using GenAI**

# **🌎Use Case :**
Planning a trip can be overwhelming and time consuming. This AI-powered notebook builds personalized, day-by-day travel itineraries based on your preferences like duration, budget etc, using Generative AI.

# **GenAI Capabilities Used :**
* **Few-shot Prompting** – use this for guiding Gemini with examples
* **Structured Output (JSON)** – clean, structured output for easy understanding
* **Embeddings (Simulated)** – match your interests, budget with attractions

# Install Required Packages

In [ ]:
!pip install -U -q "google-genai==1.7.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 4.2 MB/s eta 0:00:00


# Import Libraries

In [ ]:
import os
import json
import random
import pandas as pd
from IPython.display import display, Markdown
import google.generativeai as genai

# 🔐Setup Gemini API Key
Get your Gemini API key from Google AI Studio.

Then add it in Add-ons secret section and name it as GOOGLE_API_KEY.


In [ ]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

# 🌍Knowledge Base :

In [ ]:
city_data = [
    {"city": "Paris", "info": "The Eiffel Tower, Louvre Museum, Seine River cruises, and charming cafes."},
    {"city": "Tokyo", "info": "Shibuya Crossing, sushi bars, cherry blossoms, anime culture, and Mount Fuji."},
    {"city": "New York", "info": "Central Park, Times Square, Broadway shows, Statue of Liberty."},
    {"city": "Rome", "info": "Colosseum, Vatican City, Roman Forum, and authentic Italian food."},
    {"city": "Bangkok", "info": "Floating markets, Grand Palace, street food, and vibrant nightlife."},
    {"city": "Barcelona", "info": "Sagrada Familia, Gothic Quarter, tapas, and beaches."},
    {"city": "Istanbul", "info": "Hagia Sophia, Bosphorus cruises, bazaars, and Turkish cuisine."}
]

city_df = pd.DataFrame(city_data)

# 🧠Embed the city information

In [ ]:
def get_embedding(text):
    response = genai.embed_content(
        model="models/embedding-001",
        content=text,
        task_type="retrieval_document"
    )
    return response['embedding']

city_df['embedding'] = city_df['info'].apply(get_embedding)


# 🔍Search function for city information data

In [ ]:
def search_city_info(query, k=1):
    query_embedding = get_embedding(query)
    similarities = city_df['embedding'].apply(
        lambda e: cosine_similarity([e], [query_embedding])[0][0]
    )
    top_k = city_df.loc[similarities.nlargest(k).index]
    return top_k.iloc[0]["info"]

# ✨Few-Shot Prompting Example for Itinerary Generation

In [ ]:
few_shot_example = """
You are a travel planning assistant. Generate a JSON itinerary with this format:
{
  "city": "City Name",
  "days": [
    {"day": 1, "activities": ["Morning: ...", "Afternoon: ...", "Evening: ..."]},
    {"day": 2, "activities": ["..."]}
  ],
  "local_food": ["Dish 1", "Dish 2"],
  "packing_tips": ["Tip 1", "Tip 2"]
}
"""

# 📋Function to generate Itinerary using Gemini API

In [ ]:
import json
import google.generativeai as genai

def generate_itinerary(city, days, interest, budget):
    user_prompt = f"""
You are a travel planner. Generate a {days}-day travel itinerary for {city}, focused on {interest} experiences and a {budget} budget.

Return ONLY valid JSON in the following format and nothing else:

{{
  "city": "City Name",
  "days": [
    {{
      "day": 1,
      "activities": [
        "Activity 1",
        "Activity 2"
      ]
    }},
    {{
      "day": 2,
      "activities": [
        "Activity 3",
        "Activity 4"
      ]
    }}
  ]
}}
Make sure it is syntactically correct JSON and parsable.
    """

    model = genai.GenerativeModel(model_name="gemini-2.0-flash")
    response = model.generate_content(user_prompt)

    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print("JSON parsing failed. Raw cleaned content below:\n")
        print(raw)
        raise e

# Output display function

In [ ]:
def display_itinerary(itin):
    display(Markdown("---"))
    display(Markdown(f"### 📍 Travel Itinerary for **{itin['city']}**"))
    for day in itin["days"]:
        day_text = f"**🗓️ Day {day['day']}**\n" + "\n".join(f"- {act}" for act in day["activities"])
        display(Markdown(day_text))
        display(Markdown("---"))
    display(Markdown(f"### ✨ **Enjoy your trip to **{itin['city']}** !** ✨"))

# 🚀Example / Input city for which you want to generate travel itinerary

In [ ]:
city = "Singapore"
days = 3
interest = "culture, tourist places, food, shopping"
budget = "moderate"

itinerary = generate_itinerary(city, days, interest, budget)
display_itinerary(itinerary)

---

### 📍 Travel Itinerary for **Singapore**

**🗓️ Day 1**
- Morning: Explore Gardens by the Bay (Free outdoor gardens, consider Cloud Forest or Flower Dome - moderate cost)
- Afternoon: Hawker food lunch at Lau Pa Sat (Affordable and diverse options)
- Afternoon: Explore Merlion Park and take photos with the iconic Merlion
- Evening: Walk along the Singapore River and enjoy the vibrant Clarke Quay (budget-friendly window shopping)

---

**🗓️ Day 2**
- Morning: Visit Chinatown (Free to explore, purchase souvenirs at reasonable prices)
- Afternoon: Explore Little India (Immerse in the culture, enjoy an affordable Indian meal)
- Afternoon: Visit the National Museum of Singapore (Moderate entrance fee)
- Evening: Enjoy the Spectra light and water show at Marina Bay Sands (Free)

---

**🗓️ Day 3**
- Morning: Visit the Buddha Tooth Relic Temple and Museum (Free entry)
- Afternoon: Head to Orchard Road for window shopping and explore department stores (moderate budget for shopping)
- Afternoon: Visit Fort Canning Park for historical exploration and scenic views (Free)
- Evening: Enjoy a final Hawker Center dinner (e.g., Maxwell Food Centre) before departure

---

### ✨ **Enjoy your trip to **Singapore** !** ✨